02 - section segmentation and noise filtering

goal:
segment raw SOC2 report page text into an standardized AICPA SOC2 sections:
1. section 1- independent service auditors report 
2. section 2- management assertion
3. section 3- descriptions
4. section 4- control descriptoins and tests of operating effectiveness
5. section 5- annexure



In [7]:
# CONFIG CELL - Execution parameters and directory paths
import os
import sys
from pathlib import Path

# Dynamically locate project root
CWD = Path(os.getcwd()).resolve()
if (CWD / "data").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "data").exists():
    PROJECT_ROOT = CWD.parent
elif (CWD / "socrr" / "data").exists():
    PROJECT_ROOT = CWD / "socrr"
else:
    PROJECT_ROOT = CWD

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_LABELED_DIR = PROJECT_ROOT / "data" / "labeled"
INTERNAL_CONTROLS_DIR = PROJECT_ROOT / "data" / "internal_controls"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

for d in [DATA_INTERIM_DIR, DATA_LABELED_DIR, INTERNAL_CONTROLS_DIR, MODELS_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"[CONFIG] Project Root: {PROJECT_ROOT}")
print(f"[CONFIG] Raw Data:     {DATA_RAW_DIR}")
print(f"[CONFIG] Interim Data: {DATA_INTERIM_DIR}")

[CONFIG] Project Root: D:\System and Organization controls 2 report info extraction
[CONFIG] Raw Data:     D:\System and Organization controls 2 report info extraction\data\raw
[CONFIG] Interim Data: D:\System and Organization controls 2 report info extraction\data\interim


step 1- libaraies & defining section heading patterns

In [8]:
import json
import re
import logging
from typing import List, Dict, Any, Tuple, Optional
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("SOCRR_Segmentation")

# Precise Line-Based Section Heading Patterns
SECTION_PATTERNS = {
    "section_1_auditor_report": [
        r"^\s*(?:SECTION\s+(?:I|1)\b|INDEPENDENT\s+(?:SERVICE\s+)?AUDITOR['’]?S\s+(?:ASSURANCE\s+)?REPORT)",
        r"^\s*INDEPENDENT\s+SERVICE\s+AUDITOR['’]?S\s+REPORT"
    ],
    "section_2_management_assertion": [
        r"^\s*(?:SECTION\s+(?:II|2)\b|MANAGEMENT['’]?S\s+ASSERTION|STATEMENT\s+BY\s+(?:THE\s+)?SERVICE\s+ORGANIZATION)",
        r"^\s*ASSERTION\s+OF\s+(?:THE\s+)?MANAGEMENT"
    ],
    "section_3_system_description": [
        r"^\s*(?:SECTION\s+(?:III|3)\b|(?:SERVICE\s+ORGANIZATION['’]?S\s+)?DESCRIPTION\s+OF\s+(?:THE\s+)?SYSTEM|SYSTEM\s+DESCRIPTION)",
        r"^\s*DESCRIPTION\s+OF\s+THE\s+BOUNDARIES\s+OF\s+THE\s+SYSTEM"
    ],
    "section_4_tests_of_controls": [
        r"^\s*(?:SECTION\s+(?:IV|4)\b|CONTROL\s+DESCRIPTION\s+AND\s+TEST(?:S)?|TESTS\s+OF\s+CONTROLS\s+AND\s+RESULTS)",
        r"^\s*TRUST\s+SERVICES\s+CRITERIA,\s+RELATED\s+CONTROLS"
    ],
    "section_5_other_information": [
        r"^\s*(?:SECTION\s+(?:V|5)\b|OTHER\s+INFORMATION\s+(?:PROVIDED\s+BY\s+MANAGEMENT)?|ANNEXURE|UNAUDITED\s+INFORMATION)"
    ]
}

# Subsections of interest within Section III
SUBSECTION_PATTERNS = {
    "subservice_organizations": r"(?:SUB[\s-]?SERVICE\s+ORGANIZATION|COMPLEMENTARY\s+SUB[\s-]?SERVICE)",
    "cuecs": r"(?:COMPLEMENTARY\s+USER[\s-]ENTITY\s+CONTROLS|CUEC|USER\s+ENTITY\s+RESPONSIBILITIES)"
}

step 2 - noise filtering & header/footer cleaner

soc2 reports have feature repeating page headers and page numbers, clean these lines

In [9]:
def clean_page_noise(text: str, page_num: int) -> str:
    """Strip running headers, repeated page numbers, and separator lines."""
    lines = text.splitlines()
    cleaned = []
    
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
            
        # Filter standalone page number lines
        if re.match(r"^\d{1,4}$", stripped):
            continue
        if re.match(r"^(?:Page|PAGE)\s+\d{1,4}(?:\s+of\s+\d{1,4})?$", stripped, re.IGNORECASE):
            continue
            
        # Filter typical running header date lines
        if re.match(r"^.+?\s+\d{1,2}\s+[A-Za-z]+\s+\d{4}\s+to\s+\d{1,2}\s+[A-Za-z]+\s+\d{4}$", stripped, re.IGNORECASE):
            continue
            
        # Filter decorative lines
        if re.match(r"^[-=_~*]{3,}$", stripped):
            continue
            
        cleaned.append(stripped)
        
    return "\n".join(cleaned)

step 3 - segment report into structured sections

locating the starting page of each section using line by line inspection of the first few lines of each page

In [10]:
def segment_report(pages: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Identify section boundaries and aggregate cleaned page text per section."""
    section_starts = {}
    
    for sec_key, patterns in SECTION_PATTERNS.items():
        found_page = None
        for p in pages:
            # Skip page 1 (cover) and 2 (typical TOC)
            if p["page_num"] <= 2 and sec_key != "section_1_auditor_report":
                continue
            lines = [l.strip() for l in p["text"].splitlines() if l.strip()]
            for line in lines[:8]:
                # Ignore Table of Contents lines with dotted leaders (e.g. '... 10')
                if re.search(r"\.{4,}\s*\d+", line):
                    continue
                for pat in patterns:
                    if re.search(pat, line, re.IGNORECASE):
                        found_page = p["page_num"]
                        break
                if found_page is not None:
                    break
            if found_page is not None:
                break
        if found_page is not None:
            section_starts[sec_key] = found_page

    # Sort discovered section starts by page number
    sorted_sections = sorted(section_starts.items(), key=lambda x: x[1])
    logger.info(f"Discovered section boundary page starts: {sorted_sections}")
    
    total_pages = len(pages)
    sections_result = {}
    
    for i, (sec_name, start_pg) in enumerate(sorted_sections):
        end_pg = sorted_sections[i+1][1] - 1 if (i + 1) < len(sorted_sections) else total_pages
        
        sec_pages_text = []
        for p in pages:
            if start_pg <= p["page_num"] <= end_pg:
                cleaned_text = clean_page_noise(p["text"], p["page_num"])
                sec_pages_text.append(cleaned_text)
                
        full_sec_text = "\n\n".join(sec_pages_text)
        
        sections_result[sec_name] = {
            "page_range": [start_pg, end_pg],
            "char_count": len(full_sec_text),
            "text": full_sec_text
        }
        
    # Detect subsections within Section III (Subservices & CUECs)
    if "section_3_system_description" in sections_result:
        sec3_text = sections_result["section_3_system_description"]["text"]
        for sub_name, pat in SUBSECTION_PATTERNS.items():
            match = re.search(pat, sec3_text, re.IGNORECASE)
            sections_result[f"sub_{sub_name}"] = {
                "detected": bool(match),
                "matched_index": match.start() if match else -1
            }
            
    return sections_result

step 4 - execute segemntation on interim reports

here creating the sample_sections.json

In [11]:
page_files = list(DATA_INTERIM_DIR.glob("*_pages.json"))
print(f"Found {len(page_files)} parsed page files.")

segmentation_summary = []

for pf in page_files:
    report_id = pf.name.replace("_pages.json", "")
    with open(pf, "r", encoding="utf-8") as f:
        pages_data = json.load(f)
        
    sections = segment_report(pages_data)
    
    out_file = DATA_INTERIM_DIR / f"{report_id}_sections.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(sections, f, indent=2, ensure_ascii=False)
        
    sec_counts = {k: v["page_range"] for k, v in sections.items() if isinstance(v, dict) and "page_range" in v}
    segmentation_summary.append({
        "report_id": report_id,
        "sections_detected": list(sec_counts.keys()),
        "section_page_ranges": sec_counts
    })
    print(f"[{report_id}] Detected sections:")
    for k, rng in sec_counts.items():
        print(f"   * {k}: pages {rng[0]} - {rng[1]}")

df_seg = pd.DataFrame(segmentation_summary)
display(df_seg)

2026-09-29 14:48:25,307 [INFO] Discovered section boundary page starts: [('section_1_auditor_report', 3), ('section_2_management_assertion', 7), ('section_3_system_description', 9), ('section_4_tests_of_controls', 28), ('section_5_other_information', 116)]


2026-09-29 14:48:25,336 [INFO] Discovered section boundary page starts: [('section_1_auditor_report', 2), ('section_2_management_assertion', 4), ('section_3_system_description', 5), ('section_4_tests_of_controls', 6), ('section_5_other_information', 8)]
2026-09-29 14:48:25,344 [INFO] Discovered section boundary page starts: []


Found 3 parsed page files.
[sample_soc_report] Detected sections:
   * section_1_auditor_report: pages 3 - 6
   * section_2_management_assertion: pages 7 - 8
   * section_3_system_description: pages 9 - 27
   * section_4_tests_of_controls: pages 28 - 115
   * section_5_other_information: pages 116 - 118
[synthetic_acme_soc2] Detected sections:
   * section_1_auditor_report: pages 2 - 3
   * section_2_management_assertion: pages 4 - 4
   * section_3_system_description: pages 5 - 5
   * section_4_tests_of_controls: pages 6 - 7
   * section_5_other_information: pages 8 - 8
[synthetic_bridge_letter] Detected sections:


,report_id,sections_detected,section_page_ranges
0,sample_soc_report,"[section_1_auditor_report, section_2_managemen...","{'section_1_auditor_report': [3, 6], 'section_..."
1,synthetic_acme_soc2,"[section_1_auditor_report, section_2_managemen...","{'section_1_auditor_report': [2, 3], 'section_..."
2,synthetic_bridge_letter,[],{}


step 5 - sanity check

In [12]:
for item in segmentation_summary:
    if "bridge" in item["report_id"].lower():
        continue
    assert len(item["sections_detected"]) >= 2, f"Too few sections detected in {item['report_id']}!"
    assert (DATA_INTERIM_DIR / f"{item['report_id']}_sections.json").exists()

print("Sanity Check Passed: Section segmentation successfully written to interim storage.")

Sanity Check Passed: Section segmentation successfully written to interim storage.
